In [2]:
# load libraries
import os,sys
import squarify # pip install squarify (algorithm for treemap)
import matplotlib.pyplot as plt
import pandas as pd
from highlight_text import fig_text
import numpy as np
import plotly.express as px
from pathlib import Path

## Visualize topic distribution

- Load data

In [ ]:
data_dir = Path('/ephemeral/home/xiong/data/Fund/CSR')
all_results_df = pd.read_csv(data_dir / 'All_AIV_2008-2024_CSV_topic_identify_consolidated_with_metadata.csv')  

In [6]:
all_results_df.columns

Index(['File_Name', 'style', 'section', 'sub_section', 'bold_text',
       'italic_text', 'par', 'original_text', 'topic_labels',
       'confidence_scores', 'reasoning', 'source_file',
       'consolidated_topic_labels', 'Title', 'Series ID', 'Link',
       'Country_Name', 'Country Code', 'Year', 'income', 'ISO-3 code',
       'ISO-2 code'],
      dtype='object')

###  Overall treemap

In [9]:
collapsed_df = all_results_df.groupby(['consolidated_topic_labels'], as_index=False)['File_Name'].count()
collapsed_df.head(2)

,consolidated_topic_labels,File_Name
0,Climate Change,2151
1,Data Adequacy,3308


In [10]:
def treemap_by_topic(collapsed_df,title="Treemap of AIV Topic Distribution - LLM"):
    # Create the treemap
    fig = px.treemap(collapsed_df, 
                    path=['consolidated_topic_labels'], 
                    values='File_Name',
                    #color='Name',
                    #color_continuous_scale='RdBu',
                    width=1000, 
                    height=800,
                    title=title)
    fig.update_traces(hovertemplate='labels=%{label}<br>total_count=%{value}<extra></extra>')
    return fig

In [1]:
# fig = treemap_by_topic(collapsed_df,title="Treemap of AIV Topic Distribution")
# fig.write_image(data_dir / 'figs' / "Treemap_AIV_Topic_Distribution_LLM.png")
# fig.write_html(data_dir / 'figs' / "Treemap_AIV_Topic_Distribution_LLM.html")
# fig.show()

#### Overall distribution by overtime

In [13]:
collapsed_df = all_results_df.groupby(['Year','consolidated_topic_labels'], as_index=False)['File_Name'].count()
collapsed_df.head(2)

,Year,consolidated_topic_labels,File_Name
0,2008,Climate Change,20
1,2008,Data Adequacy,285


In [14]:
def time_series_by_topic(collapsed_df,title='Topic Over Time - LLM'):
    fig = px.area(collapsed_df, 
                x="Year", 
                y="File_Name", 
                color="consolidated_topic_labels", 
                line_group="consolidated_topic_labels",
                height=500,
                width=1000,
                title=title,
                )
    fig.update_layout(legend=dict(
                                title=dict(
                                    text='High Level Topics',
                                    font=dict(size=14)  # Optional: set the font size
                                ),),
                        yaxis_title='# of paragraphs'
        
                        )
    return fig

In [2]:
# fig = time_series_by_topic(collapsed_df)
# fig.write_image(data_dir / 'figs' / "TimeSeries_AIV_Topic_counts_LLM.png")
# fig.write_html(data_dir / 'figs' / "TimeSeries_AIV_Topic_counts_LLM.html")
# fig.show()

#### Topic distribution by year by percentage

In [16]:
collapsed_df = all_results_df.groupby(['Year','consolidated_topic_labels'], as_index=False)['File_Name'].count()
collapsed_df['File_Name'] = collapsed_df.groupby('Year')['File_Name'].transform(lambda x: (x / x.sum()*100))
collapsed_df.head(2)

,Year,consolidated_topic_labels,File_Name
0,2008,Climate Change,0.160346
1,2008,Data Adequacy,2.284935


In [17]:
def time_series_level_1_percentage(collapsed_df, title = 'Topic Distribution Over Time - LLM'):
    fig = px.area(collapsed_df, 
                x="Year", 
                y="File_Name", 
                color="consolidated_topic_labels", 
                line_group="consolidated_topic_labels",
                height=500,
                width=1000,
                title=title,
                )
    fig.update_layout(legend=dict(
                                title=dict(
                                    text='High Level Topics',
                                    font=dict(size=14)  # Optional: set the font size
                                ),),
                            yaxis=dict(
                                        title='Percentage',  # Optional: Set the y-axis title
                                        range=[0, 100]  # Set the y-axis range from 0 to 100
                                    ),
        
                        )
    return fig


In [3]:
# fig = time_series_level_1_percentage(collapsed_df)
# fig.write_image(data_dir / 'figs' / "TimeSeries_AIV_Topic_Distribution_LLM.png")
# fig.write_html(data_dir / 'figs' / "TimeSeries_AIV_Topic_Distribution_LLM.html")
# fig.show()